<a href="https://colab.research.google.com/github/JH98765432/J/blob/main/0403.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

# 1. 페이지 접속
url = "https://books.toscrape.com/"
response = requests.get(url)
soup = BeautifulSoup(response.text, 'html.parser')

# 2. 데이터 추출 (h3 -> a의 title 속성)
titles = [h3.find('a')['title'] for h3 in soup.find_all('h3')]

# 3. 데이터프레임 생성 (1~20번)
df_20 = pd.DataFrame(titles, columns=['Book Title'])
print("--- 1~20개 결과 ---")
print(df_20)

--- 1~20개 결과 ---
                                           Book Title
0                                A Light in the Attic
1                                  Tipping the Velvet
2                                          Soumission
3                                       Sharp Objects
4               Sapiens: A Brief History of Humankind
5                                     The Requiem Red
6   The Dirty Little Secrets of Getting Your Dream...
7   The Coming Woman: A Novel Based on the Life of...
8   The Boys in the Boat: Nine Americans and Their...
9                                     The Black Maria
10     Starving Hearts (Triangular Trade Trilogy, #1)
11                              Shakespeare's Sonnets
12                                        Set Me Free
13  Scott Pilgrim's Precious Little Life (Scott Pi...
14                          Rip it Up and Start Again
15  Our Band Could Be Your Life: Scenes from the A...
16                                               Olio
17  Mesaeri

In [2]:
all_titles = []

for i in range(1, 6): # 1페이지부터 5페이지까지
    page_url = f"https://books.toscrape.com/catalogue/page-{i}.html"
    res = requests.get(page_url)
    s = BeautifulSoup(res.text, 'html.parser')

    # 각 페이지의 제목들을 리스트에 추가
    titles = [h3.find('a')['title'] for h3 in s.find_all('h3')]
    all_titles.extend(titles)

# 데이터프레임 생성
df_100 = pd.DataFrame(all_titles, columns=['Book Title'])
print(f"총 수집된 제목 수: {len(df_100)}")
print(df_100.head())

총 수집된 제목 수: 100
                              Book Title
0                   A Light in the Attic
1                     Tipping the Velvet
2                             Soumission
3                          Sharp Objects
4  Sapiens: A Brief History of Humankind


In [3]:
import re # 가격에서 숫자만 추출하기 위한 정규표현식

categories = [
    'travel_2', 'mystery_3', 'historical-fiction_4', 'sequential-art_5', 'classics_6',
    'philosophy_7', 'romance_8', 'womens-fiction_9', 'fiction_10', 'childrens_11'
]

category_data = []

for cat in categories:
    cat_url = f"https://books.toscrape.com/catalogue/category/books/{cat}/index.html"
    res = requests.get(cat_url)
    s = BeautifulSoup(res.text, 'html.parser')

    # 해당 카테고리의 책 리스트 (최대 10개만 수집)
    products = s.find_all('article', class_='product_pod')[:10]

    for product in products:
        title = product.h3.a['title']
        price_text = product.find('p', class_='price_color').text
        # 가격 문자열에서 숫자와 소수점만 남기기 (예: £51.77 -> 51.77)
        price = float(re.findall(r"\d+\.\d+", price_text)[0])

        category_data.append({'Category': cat.split('_')[0], 'Title': title, 'Price': price})

# 데이터프레임 생성
df_final = pd.DataFrame(category_data)

# 가격(Price) 기준으로 내림차순 정렬 후 상위 10개 추출
top_10_expensive = df_final.sort_values(by='Price', ascending=False).head(10)

print("--- 가장 비싼 책 TOP 10 ---")
print(top_10_expensive)

--- 가장 비싼 책 TOP 10 ---
              Category                                              Title  \
46            classics                                            Candide   
51          philosophy       The Death of Humanity: and the Case for Life   
93           childrens  The White Cat and the Monk: A Retelling of the...   
70      womens-fiction  I Had a Nice Time And Other Lies...: How to fi...   
47            classics                                        Animal Farm   
7               travel                   A Year in Provence (Provence #1)   
12             mystery                                The Past Never Ends   
92           childrens                    The Secret of Dreadwillow Carse   
66             romance                   Suddenly in Love (Lake Haven #1)   
22  historical-fiction            A Flight of Arrows (The Pathfinders #2)   

    Price  
46  58.63  
51  58.11  
93  58.08  
70  57.36  
47  57.22  
7   56.88  
12  56.50  
92  56.13  
66  55.99  
22  55.53